In [1]:
!pip install plotly

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 15.6/15.6 MB 29.3 MB/s eta 0:00:0000:0100:01


In [1]:
from samap.mapping import SAMAP
from samap.analysis import (get_mapping_scores, GenePairFinder, transfer_annotations,
                            sankey_plot, chord_plot, CellTypeTriangles, 
                            ParalogSubstitutions, FunctionalEnrichment,
                            convert_eggnog_to_homologs, GeneTriangles)
from samalg import SAM
import pandas as pd
from Bio import SeqIO
from samap.utils import (save_samap, load_samap)
import scanpy as sc
import matplotlib.colors
import matplotlib.pyplot as plt
import numpy as np
from scipy import stats
from scipy import sparse 
from scipy import cluster
import seaborn as sns
import random
import sklearn
from scipy.stats import poisson
from sklearn.neighbors import KernelDensity
import time
import dill
from scipy.optimize import minimize
import pickle
import itertools
import os
import plotly.express as px
import time
from sklearn.metrics.cluster import adjusted_rand_score

/scratch/miniconda/lib/python3.7/site-packages/tqdm/auto.py:22: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [33]:
fn = '../../Subclustering/MNPO/SAM_CJ_joined_v2_cleaned_08182025_MNPO.h5ad'
sam = SAM()
sam.load_data(fn)

In [34]:
sam_test = SAM()
sam_test.load_data('../../Active_SAM_joined/SAM_CJ_joined_v2_cleaned_03122025.h5ad')

In [35]:
sam_test.adata.obs.columns

Index(['orig.ident', 'nCount_RNA', 'nFeature_RNA', 'n_genes', 'n_counts',
       'key', 'hicat_merged', 'subclass_id_label_mapping',
       'subclass_id_label_lc', 'leiden_clusters',
       'subclass_id_label_mapping_nounlabeled', 'neurotransmitter',
       'region_label', 'subclass_id_label_reduced_mapping',
       'subclass_id_label_reduced_lc',
       'subclass_id_label_reduced_mapping_nounlabeled', 'fraction_match',
       'best_match', 'frac_match_test', 'leiden_removal', 'nCount_SCT',
       'nFeature_SCT', 'SCT_snn_res.0.8', 'seurat_clusters', 'SCT_snn_res.5',
       'eq_subclass', 'eq_subclass_lc', 'eq_subclass_frac',
       'eq_subclass_nounlabeled', 'eq_subclass_nounlabeled_NN',
       'eq_subclass_nounlabeled_nmm', 'ss_subclass', 'ss_subclass_nounlabeled',
       'ss_class', 'ss_subclass_nounlabeled_astro', 'ss_subclass_v2',
       'ss_subclass_v2_nounlabeled', 'ss_subclass_nounlabeled_nmm',
       'ss_subclass_v3_nounlabeled', 'subclass_id_label_crossed',
       'ss_subclas

In [36]:
tester = sam_test.adata.X[sam_test.adata.obs['ss_subclass'] == '116 AVPV-MEPO-SFO Tbr1 Glut']

In [37]:
tester

<421x20032 sparse matrix of type '<class 'numpy.float32'>'
	with 476841 stored elements in Compressed Sparse Row format>

In [38]:
sam.adata.X

<421x20032 sparse matrix of type '<class 'numpy.float32'>'
	with 476841 stored elements in Compressed Sparse Row format>

In [39]:
def csr_equal(A, B):
    if A.shape != B.shape:
        return False
    return (A != B).nnz == 0

In [40]:
csr_equal(tester,sam.adata.X)

True

In [41]:
sam.clustering(param =0.5)

In [11]:
sm = load_samap('../../Subclustering/MNPO/sm_hypoorgs_MNPO_directsub_08092026.pkl')

In [12]:
test_sm = load_samap('../../Subclustering/MNPO/sm_hypoorgs_MNPO_08042026.pkl')

In [ ]:
test_sm.sams['mg'].adata.obs.columns

In [15]:
sm.sams['mg'].adata.obs['eq_supertype_cl_v2'] = test_sm.sams['mg'].adata.obs['eq_supertype_cl_v2']

In [42]:
org_two = 'cj'

In [56]:
sm.sams[org_two].clustering(param =0.5)

In [ ]:
sm.sams[org_two].adata.obs.columns

In [58]:
keys = {'mg':'eq_supertype_cl_v2',org_two:'leiden_clusters'}
D,MappingTable = get_mapping_scores(sm,keys)

lim_MappingTable = MappingTable.filter(like='mg_')
lim_MappingTable = lim_MappingTable[lim_MappingTable.index.str.contains(org_two + '_')]

/scratch/miniconda/lib/python3.7/site-packages/samap/analysis.py:1609: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead.  To get a de-fragmented frame, use `newframe = frame.copy()`
  samap.adata.obs[l] = pd.Categorical(cl)


In [ ]:
lim_MappingTable

In [ ]:
csr_equal(sam.adata.X,sm.sams[org_two].adata.X)

In [ ]:
sm.sams['mg'].adata.obs

In [62]:
fin_mapping = {}
for item in lim_MappingTable.index:
    bm = lim_MappingTable.loc[item,:].idxmax()
    if lim_MappingTable.loc[item,bm] > .2:
        fin_mapping[item] = bm
    else:
        fin_mapping[item] = item
        
fin_labels = []
for item in sm.sams[org_two].adata.obs['leiden_clusters']:
    fin_labels.append(fin_mapping[org_two + '_' + str(item)][3:])

In [63]:
sam.adata.obs['fin_test'] = fin_labels

In [108]:
mg_genes = ['Etv1','Onecut1','Bcl11a','Foxp2','Ebf1','Emx2','Adcyap1','Agtr1a','Rxfp1','Crh','Trpc3','Sncg', 'Opn5','Rxfp2','Npy','Brs3','Ucn3','Slc2a9']
mo_genes = ['Etv1','Onecut1','Bcl11a','Foxp2','Ebf1','Emx2','Adcyap1','ENSMOCG00000019499','Rxfp1','Crh','Trpc3','Sncg', 'Opn5','Rxfp2','Npy','Brs3','UCN3','Slc2a9']
cj_genes = ['ETV1','ONECUT1','BCL11A','FOXP2','EBF1','EMX2','ADCYAP1','AGTR1','RXFP1','CRH','TRPC3','SNCG','OPN5','ENSCJPG00005003031','NPY','BRS3','SLC2A9']
ac_genes = ['etv1','bcl11a','ebf1','emx2','adcyap1','rxfp1','crh','trpc3','sncg','LOC100554017','LOC100553533','npy','brs3','ucn3','slc2a9']
ri_genes = ['LOC138641708','ONECUT1','BCL11A','FOXP2','EBF1','EMX2','ADCYAP1','LOC138638365','RXFP1','CRH','TRPC3','SNCG', 'OPN5','RXFP2','NPY','BRS3','UCN3','SLC2A9']

In [ ]:
sc.pl.dotplot(sam.adata,groupby = 'fin_test',var_names=ri_genes)

In [ ]:
sam.adata.obs

In [75]:
fin_labels = []
for item in sm.sams[org_two].adata.obs['leiden_clusters']:
    fin_labels.append(fin_mapping[org_two + '_' + str(item)][3:])

In [76]:
sam.adata.obs['eq_supertype_cl_v4'] = fin_labels
sm.sams[org_two].adata.obs['eq_supertype_cl_v4'] = fin_labels

In [117]:
sam.adata.obs['eq_clusters_v4'] = sm.sams[org_two].adata.obs['leiden_clusters']
sm.sams[org_two].adata.obs['eq_clusters_v4'] = sm.sams[org_two].adata.obs['leiden_clusters']

In [77]:
sam.save_anndata(fn)

In [92]:
save_samap(sm,'../../Subclustering/MNPO/sm_hypoorgs_MNPO_08042026.pkl')